# 전기차 에너지 소비량 예측 기반 기업 운행 효율 분석 프로젝트

## 1. 문서 목적

이 노트북은 전기차 주행 데이터를 분석하여 **에너지 소비량 예측 모델을 구축하고**, 기업이 다양한 운행 조건에 따른 예상 소비량을 비교하여 효율적인 운행 계획을 검토할 수 있도록 하는 실습 프로젝트의 방향을 정리한다.

이번 프로젝트는 실습 기간과 현재 데이터의 한계를 고려하여 개인 운전자별 장기 분석, 경로·배차 최적화, 예지정비 등으로 범위를 넓히지 않고 다음 세 가지에 집중한다.

1. **에너지 소비량 예측**
2. **운행 조건별 소비량 시뮬레이션**
3. **에너지 소비량과 관련성이 높은 주요 요인 분석**

> 핵심 목표는 단순히 소비량을 예측하는 데서 끝나지 않고, 예측 모델을 활용해 기업이 여러 운행 조건을 비교할 수 있도록 하는 것이다.


## 2. 데이터셋 개요

현재 데이터셋은 총 **8,000개 주행 기록과 10개 수치형 변수**로 구성되어 있으며, 결측치와 완전히 중복된 행은 없는 것으로 정리되어 있다.

예측 대상은 `energy_consumption_kwhper100km`이며, **100km 주행에 필요한 에너지 소비량**을 의미한다. 연속형 값을 예측하므로 지도학습의 **회귀 문제**로 정의한다.

| 구분 | 변수 | 의미 | 관측 범위 |
|---|---|---|---:|
| 입력 | `speed_kmh` | 평균 주행 속도로 해석 | 20.0~130.0 km/h |
| 입력 | `payload_kg` | 탑승자 및 화물 하중 | 0~500 kg |
| 입력 | `ambient_temp_C` | 외부 기온 | -10~40°C |
| 입력 | `hvac_power_kw` | 냉난방 사용 전력 | 0~5 kW |
| 입력 | `road_grade_pct` | 도로 경사도 | -5~8% |
| 입력 | `battery_temp_C` | 배터리 온도 | 15~45°C |
| 입력 | `driving_style_index` | 운전 스타일 지수 | 0~1 |
| 입력 | `tire_pressure_bar` | 타이어 공기압 | 2.0~2.8 bar |
| 입력 | `trip_distance_km` | 주행 거리 | 5.1~200 km |
| 타깃 | `energy_consumption_kwhper100km` | 100km당 에너지 소비량 | 11.62~35.00 kWh/100km |

### 데이터 해석 시 주의

- `speed_kmh`는 데이터 구조상 한 번의 주행을 대표하는 **평균 또는 대표 주행 속도**로 해석하는 것이 자연스럽지만, 원본 설명에서 평균값이라고 명시된 것은 아니다.
- `driving_style_index`는 구체적인 산정 방식이 제공되지 않았으므로 지수의 절대적인 의미를 단정하지 않는다.
- 현재 데이터에는 `driver_id`, `vehicle_id`, 날짜·시간 정보가 없어 특정 운전자나 특정 차량의 연속 주행 기록으로 볼 수 없다.


## 3. 프로젝트 범위

### 이번 프로젝트에서 구현할 범위

#### 1) 에너지 소비량 예측
주행 조건을 입력하면 예측 모델이 `kWh/100km` 단위의 예상 에너지 소비량을 반환한다.

#### 2) 운행 조건별 시뮬레이션
속도, HVAC 사용 전력, 적재량 등 일부 조건을 변경해 여러 시나리오의 예상 소비량을 비교한다.

예시:

| 시나리오 | 평균 속도 | HVAC | 적재량 | 예상 소비량 |
|---|---:|---:|---:|---:|
| 현재 계획 | 100 km/h | 3.0 kW | 400 kg | 26.2 kWh/100km |
| 대안 A | 90 km/h | 2.5 kW | 400 kg | 23.9 kWh/100km |

이와 같이 현재 계획과 대안을 비교하여 **예상 절감 가능성**을 보여준다.

#### 3) 에너지 소비량 관련 요인 분석
상관관계, 변수 중요도, 순열 중요도 등 모델 해석 방법을 이용하여 어떤 주행 조건이 높은 에너지 소비량 예측과 관련되어 있는지 확인한다.

### 이번 프로젝트에서 제외할 범위

현재 데이터와 실습 기간을 고려하여 다음 기능은 구현 범위에서 제외한다.

- 개인 운전자별 일간·주간·월간 분석
- 운전자별 또는 차량별 장기 비교
- 실시간 경로 추천 및 최적화
- 여러 차량의 배차·화물 분배 최적화
- 충전소 및 충전 스케줄 최적화
- 차량 고장 예측 및 예지정비

이 기능들은 추가 데이터가 확보되었을 때 확장 가능한 주제로 남긴다.


## 4. 입력 변수를 기업 서비스 관점에서 분류

모든 입력 변수를 자유롭게 변경할 수 있는 것은 아니다. 서비스에서는 **변경 가능한 조건**, **간접적으로 선택 가능한 조건**, **고정되는 외부 조건**, **안전이 우선되는 상태 변수**를 구분해야 한다.

| 구분 | 변수 | 서비스에서의 역할 |
|---|---|---|
| 외부 조건 | 외부 기온 | 사용자가 변경할 수 없으므로 시뮬레이션에서는 주어진 조건으로 사용 |
| 간접 선택 조건 | 도로 경사도 | 경사 자체는 변경할 수 없으나 향후 경로 후보가 있다면 비교 가능 |
| 운행 조건 | 평균 속도 | 운행 계획에서 조정 가능한 핵심 변수 |
| 운행 조건 | HVAC 사용 전력 | 승객 쾌적성과 안전을 해치지 않는 범위에서 비교 가능 |
| 배차·운행 계획 | 적재량 | 화물·탑승자 하중에 따른 소비량 변화 비교 가능 |
| 배차·운행 계획 | 주행 거리 | 운행 계획에서 주어진 값으로 활용 |
| 상태 변수 | 배터리 온도 | 자유로운 최적화 대상보다는 상태 확인에 활용 |
| 상태 변수 | 타이어 공기압 | 제조사 권장 범위가 우선되는 관리 변수 |
| 해석 주의 변수 | 운전 스타일 지수 | 소비량과의 관계는 분석할 수 있으나 구체적인 산정 기준이 없어 추천 변수로는 보수적으로 사용 |

### 핵심 원칙

> **예측에 중요한 변수**와 **기업이 실제로 조절할 수 있는 변수**는 서로 다를 수 있다.

따라서 모델에서 중요도가 높게 나타났다고 해서 모든 변수를 직접 변경 대상으로 추천해서는 안 된다.


## 5. 현재 데이터에서 확인된 주요 관계

에너지 소비량과 각 입력 변수 간 피어슨 상관계수는 다음과 같이 정리되어 있다.

| 변수 | 에너지 소비량과의 상관계수 | 1차 해석 |
|---|---:|---|
| 도로 경사도 | +0.505 | 경사도가 증가할수록 소비량이 증가하는 경향 |
| 적재량 | +0.469 | 하중이 증가할수록 소비량이 증가하는 경향 |
| 평균 속도 | +0.448 | 높은 속도에서 소비량이 증가하는 경향 |
| HVAC 사용 전력 | +0.370 | 냉난방 사용 전력이 증가할수록 소비량이 증가하는 경향 |
| 운전 스타일 지수 | +0.258 | 지수가 높을수록 소비량이 증가하는 경향 |
| 주행 거리 | +0.172 | 비교적 약한 양의 관계 |
| 외부 기온 | -0.155 | 현재 데이터에서는 약한 음의 관계 |
| 타이어 공기압 | -0.095 | 약한 음의 관계 |
| 배터리 온도 | -0.087 | 약한 음의 관계 |

### 해석 시 주의

- 상관계수는 두 변수의 **선형적인 관계**를 요약한 값이다.
- 상관계수가 낮다고 해서 반드시 관계가 없는 것은 아니다.
- 특히 온도처럼 비선형 또는 U자 형태의 관계가 있을 수 있는 변수는 산점도, 구간별 평균, 모델 기반 분석을 추가로 확인해야 한다.
- 상관관계만으로 특정 변수가 에너지 소비 증가의 직접적인 원인이라고 단정할 수 없다.


## 6. 프로젝트 문제 정의

### 업무 관점의 문제

전기차의 에너지 소비량은 속도, 적재량, 기온, 냉난방 사용량, 도로 경사 등 다양한 조건에 따라 달라질 수 있다.

기업이 운행 전에 예상 소비량을 확인할 수 있다면 다음과 같은 의사결정에 참고할 수 있다.

- 현재 운행 계획에서 예상되는 에너지 소비량 확인
- 여러 운행 조건의 예상 소비량 비교
- 높은 소비량과 관련성이 큰 조건 파악
- 운행 조건 변경에 따른 예상 절감 가능성 검토

### 예측 문제

입력된 주행·환경 조건을 이용하여 다음 값을 예측한다.

**예측 대상:** `energy_consumption_kwhper100km`

즉, 특정 운행 조건에서 **100km당 예상 에너지 소비량**을 예측한다.

### 프로젝트 정의 문장

> 본 프로젝트는 전기차의 주행 속도, 적재량, 외부 기온, HVAC 사용 전력, 도로 경사도, 배터리 온도, 운전 스타일 지수, 타이어 공기압 및 주행 거리를 이용하여 에너지 소비량을 예측하고, 기업이 여러 운행 조건의 예상 소비량을 비교하여 효율적인 운행 계획을 검토할 수 있도록 하는 것을 목표로 한다.


## 7. 서비스 아이디어: 기업용 EV 운행 효율 분석 도구

서비스의 기본 흐름은 다음과 같다.

**운행 조건 입력 → 에너지 소비량 예측 → 조건별 시뮬레이션 → 결과 비교 → 주요 관련 요인 설명**

### 입력 예시

- 계획 평균 속도
- 탑승자 및 화물 하중
- 외부 기온
- HVAC 사용 전력
- 도로 경사도
- 배터리 온도
- 운전 스타일 지수
- 타이어 공기압
- 주행 거리

### 출력 예시

| 항목 | 결과 예시 |
|---|---:|
| 예상 에너지 소비량 | 25.8 kWh/100km |
| 120km 주행 시 예상 총소비 에너지 | 약 31.0 kWh |
| 대안 시나리오 예상 소비량 | 23.9 kWh/100km |
| 현재 계획 대비 예상 절감률 | 약 7.4% |

### 총소비 에너지 계산

예측된 소비량이 `25.8 kWh/100km`, 주행 거리가 `120km`라면:

**예상 총소비 에너지 = 25.8 × 120 / 100 = 약 31.0 kWh**

`kWh/100km`는 모델이 직접 예측하고, 특정 거리의 총소비 에너지는 예측 결과와 주행 거리를 이용해 계산한다.


## 8. 핵심 서비스 기능

### 8.1 에너지 소비량 예측

기업 담당자가 주행 조건을 입력하면 모델이 해당 조건에서의 예상 에너지 소비량을 반환한다.

예시:

- 평균 속도: 90 km/h
- 적재량: 350 kg
- 외부 기온: 8°C
- HVAC: 2.5 kW
- 도로 경사도: 2%
- 주행 거리: 120 km

**예상 결과:** 24.3 kWh/100km

이 기능이 프로젝트의 핵심 머신러닝 기능이다.

---

### 8.2 운행 조건별 소비량 시뮬레이션

다른 조건을 고정한 상태에서 조절 가능한 변수의 값을 변경해 예상 소비량을 반복 비교한다.

예를 들어 속도를 비교한다면:

| 평균 속도 | 예상 소비량 |
|---:|---:|
| 70 km/h | 22.4 kWh/100km |
| 80 km/h | 23.1 kWh/100km |
| 90 km/h | 24.6 kWh/100km |

또는 현재 계획과 변경된 계획을 함께 비교할 수 있다.

| 항목 | 현재 계획 | 대안 |
|---|---:|---:|
| 평균 속도 | 100 km/h | 90 km/h |
| HVAC | 3.0 kW | 2.5 kW |
| 적재량 | 400 kg | 400 kg |
| 예상 소비량 | 26.2 | 23.9 |

**해석 예시:**  
`현재 조건과 비교했을 때 대안 조건에서는 에너지 소비량이 약 8.8% 낮게 예측됩니다.`

이 결과는 실제 절감 효과를 보장하는 값이 아니라 **모델 기반 시뮬레이션 결과**로 표현한다.

---

### 8.3 에너지 소비량 관련 요인 분석

EDA와 모델 해석 결과를 이용해 소비량 예측에 관련성이 높은 변수를 확인한다.

활용 가능한 방법:

- 상관관계 분석
- 선형회귀 계수
- 트리 기반 모델의 변수 중요도
- 순열 중요도
- 필요 시 PDP, ICE, SHAP 등의 추가 분석

서비스에서는 다음과 같이 설명할 수 있다.

> 현재 운행 조건에서는 높은 평균 속도와 HVAC 사용 전력이 높은 에너지 소비량 예측과 관련되어 있습니다.

단, **'속도 때문에 소비량이 증가했다'와 같이 인과관계로 단정하지 않는다.**


## 9. 예측 모델과 시뮬레이션의 관계

### 예측 모델의 역할

예측 모델은 하나의 주행 조건을 입력받아 예상 에너지 소비량을 계산한다.

**주행 조건 → 예측 모델 → 예상 kWh/100km**

### 시뮬레이션의 역할

시뮬레이션은 예측 모델 자체와 별개의 새로운 모델이라기보다, **기존 예측 모델에 여러 후보 조건을 반복 입력하여 결과를 비교하는 기능**이다.

예:

1. 현재 계획의 예상 소비량 계산
2. 평균 속도를 변경한 대안 계산
3. HVAC 사용 전력을 변경한 대안 계산
4. 여러 조건을 함께 변경한 대안 계산
5. 현재 계획과 각 대안의 예상 소비량 비교

### 추천 표현

현재 프로젝트에서는 복잡한 최적화 시스템보다는 다음 표현이 더 적절하다.

- `효율적인 운행 조건 후보`
- `조건별 예상 소비량 비교`
- `현재 계획 대비 예상 절감량`
- `모델 기반 시뮬레이션`

현재 데이터만으로 실제 운행의 최적 조건을 확정적으로 보장할 수는 없다.


## 10. 머신러닝 모델링 계획

### 모델 후보

1. **평균값 예측 모델**  
   최소 기준선으로 사용

2. **Linear Regression**  
   기준 모델로 사용하며 변수와 소비량의 선형 관계를 해석하기 쉬움

3. **KNN Regressor**  
   거리 기반 회귀 모델의 특성과 스케일링 영향을 비교하는 실습에 활용 가능

4. **Random Forest Regressor**  
   비선형 관계와 변수 간 상호작용을 반영할 수 있음

5. **Gradient Boosting 계열 모델**  
   예측 성능 개선 후보로 비교 가능

### 평가 지표

- **MAE**  
  평균적으로 몇 `kWh/100km` 정도의 오차가 발생하는지 직관적으로 확인

- **RMSE**  
  큰 예측 오차에 더 큰 패널티를 부여

- **R²**  
  에너지 소비량 변동을 모델이 어느 정도 설명하는지 확인

### 모델 선정 원칙

단순히 Train 성능이 가장 높은 모델을 선택하지 않고 다음을 함께 고려한다.

- Test 성능
- 교차검증 결과
- Train과 Test 성능 차이
- 과적합 여부
- 모델 해석 가능성
- 시뮬레이션에 사용하기 적절한 안정성

필요한 경우 GridSearchCV 등을 이용해 주요 하이퍼파라미터를 튜닝한다.


## 11. 모델 해석 계획

예측 모델을 서비스에 활용하려면 단순히 성능 점수만 제시하는 것보다 **왜 그런 예측이 나왔는지 설명할 수 있는 근거**가 필요하다.

### 기본 분석

- 각 변수와 타깃 간 상관관계
- 변수별 분포
- 변수와 소비량의 산점도
- 구간별 평균 소비량

### 모델 기반 분석

#### 선형 모델
회귀계수의 부호와 크기를 참고하여 변수 변화와 예측값의 관계를 확인한다.

#### 트리 기반 모델
Feature Importance를 이용하여 어떤 변수가 예측에 많이 활용되는지 확인한다.

#### 추가 분석
프로젝트 진행 상황에 따라 다음 방법을 선택적으로 사용한다.

- Permutation Importance
- Partial Dependence Plot
- ICE
- SHAP

### 해석 원칙

모델 해석 결과는 다음과 같이 표현한다.

**권장 표현:**  
`높은 평균 속도가 높은 소비량 예측과 관련되어 있습니다.`

**피해야 할 표현:**  
`평균 속도가 높아서 소비량이 증가했습니다.`

현재 프로젝트는 관측 데이터 기반 예측이므로 인과관계를 직접 증명하는 것은 아니다.


## 12. 주요 변수 사용 시 주의사항

### 12.1 `driving_style_index`

데이터상으로는 지수가 높을수록 에너지 소비량도 증가하는 양의 관계가 관찰되지만, 지수의 구체적인 산정 방식은 제공되지 않았다.

따라서:

- 모델의 입력 Feature로는 활용 가능
- 상관관계 및 중요도 분석 가능
- `0.7에서 0.4로 낮추세요`와 같은 행동 추천에는 주의
- 구체적인 운전 습관 개선 기능에는 급가속·급제동 등 추가 데이터 필요

---

### 12.2 `speed_kmh`

현재 데이터 구조에서는 평균 또는 대표 주행 속도로 해석하는 것이 자연스럽다.

운행 전 서비스에서는 실제 평균 속도가 아직 결정되지 않았으므로 **계획 평균 속도**로 사용한다는 가정이 필요하다.

---

### 12.3 타이어 공기압

모델에서 공기압에 따른 소비량 차이가 나타난다고 하더라도 에너지 절감을 이유로 임의의 공기압을 추천해서는 안 된다.

**차량 및 타이어 제조사의 권장 범위가 우선**이다.

---

### 12.4 배터리 온도

현재 타깃은 에너지 소비량이므로 배터리 온도를 이용해 고장이나 안전 위험을 직접 판단할 수 없다.

배터리 온도는 현재 프로젝트에서는 **에너지 소비량 예측에 사용되는 상태 변수**로 한정한다.


## 13. 해석 및 서비스 적용 시 주의사항

### 13.1 상관관계와 인과관계 구분

모델은 데이터에서 관측된 관계를 학습한다. 특정 조건을 변경하면 실제 소비량이 반드시 예측한 만큼 변한다고 보장할 수 없다.

따라서 다음 표현을 사용한다.

- 예상 소비량
- 예상 절감량
- 모델 기반 비교
- 시뮬레이션 결과
- 소비량과 관련성이 높은 요인

---

### 13.2 학습 데이터 범위 밖의 값 제한

현재 데이터의 범위를 크게 벗어나는 조건은 모델이 충분히 학습하지 않은 영역이므로 예측 신뢰도가 낮아질 수 있다.

시뮬레이션 후보는 가능하면 학습 데이터 범위 안에서 생성한다.

---

### 13.3 실제 서비스와 실습 프로젝트 구분

현재 데이터에는 다음 정보가 없다.

- 차량 ID
- 운전자 ID
- 시간 정보
- 배터리 용량 및 SOC
- 차량 모델 및 제원
- 실제 경로
- 충전 기록
- 고장 및 정비 이력

따라서 이번 결과물은 실제 상용 Fleet 관리 시스템이 아니라 **기업 관점의 에너지 소비 예측 및 운행 조건 비교를 구현하는 실습용 개념 검증**으로 정의한다.


## 14. 프로젝트 진행 흐름

프로젝트는 다음 순서로 진행한다.

### 1단계. 데이터 이해
- 컬럼 의미 확인
- 데이터 크기 및 자료형 확인
- 결측치·중복값 확인
- 데이터 범위 확인

### 2단계. EDA
- 각 변수의 분포 확인
- 이상치 확인
- 타깃과 각 변수의 관계 확인
- 상관관계 분석

### 3단계. 전처리
- 필요 시 스케일링
- Train/Test 데이터 분리
- 모델에 맞는 Pipeline 구성 검토

### 4단계. 회귀 모델 비교
- 기준선 모델
- Linear Regression
- KNN
- Random Forest
- Gradient Boosting 계열 등

### 5단계. 교차검증 및 튜닝
- K-Fold 또는 적절한 교차검증
- GridSearchCV 등을 이용한 하이퍼파라미터 비교
- MAE, RMSE, R² 평가

### 6단계. 최종 모델 선정
- 성능
- 과적합 여부
- 안정성
- 해석 가능성

### 7단계. 모델 해석
- 변수 중요도
- 순열 중요도 등
- 소비량과 관련성이 높은 조건 정리

### 8단계. 기업용 시뮬레이션 구성
- 현재 운행 계획 입력
- 예상 소비량 출력
- 대안 조건 생성
- 조건별 예상 소비량 비교
- 현재 계획 대비 예상 절감량 제시


## 15. 최종 결과 화면 예시

### 현재 운행 계획

| 항목 | 값 |
|---|---:|
| 평균 속도 | 100 km/h |
| 적재량 | 400 kg |
| HVAC 사용 전력 | 3.0 kW |
| 예상 에너지 소비량 | 26.2 kWh/100km |

### 대안 운행 조건

| 항목 | 값 |
|---|---:|
| 평균 속도 | 90 km/h |
| 적재량 | 400 kg |
| HVAC 사용 전력 | 2.5 kW |
| 예상 에너지 소비량 | 23.9 kWh/100km |

### 비교 결과

- 현재 계획 대비 예상 소비량 감소: **약 8.8%**
- 120km 주행 시 예상 총소비 에너지 감소량도 함께 계산 가능
- 결과는 확정적인 절감 효과가 아니라 **모델 기반 예상값**으로 제공

### 관련 요인 예시

> 현재 계획에서는 평균 속도와 HVAC 사용 전력이 높은 소비량 예측과 관련된 주요 조건으로 나타났습니다.

이 결과 화면은 예측값 하나만 보여주는 것이 아니라, **현재 조건 → 대안 조건 → 예상 변화 → 관련 요인**의 흐름으로 구성한다.


## 16. 향후 확장 가능 기능

다음 기능은 이번 실습 범위에는 포함하지 않지만 추가 데이터가 확보될 경우 확장 가능하다.

### 차량별 운영 분석
필요 데이터:
- `vehicle_id`
- 차량 모델
- 배터리 용량
- SOC
- 차량별 운행 기록

### 경로별 소비량 비교
필요 데이터:
- 출발지·목적지
- 실제 도로 거리
- 경로별 평균 속도
- 경로별 경사도
- 교통 정보

### 차량 배차 및 화물 분배 최적화
필요 데이터:
- 차량 ID
- 차량별 최대 적재량
- 배송지
- 주문·화물 정보
- 시간 제약

### 예지정비 및 이상 탐지
필요 데이터:
- 시계열 센서 데이터
- 실제 소비량
- 고장 이력
- 정비 이력
- 배터리 SOH 및 셀 상태

현재 프로젝트에서는 이러한 기능을 실제 구현하지 않고 **확장 아이디어로만 정리한다.**


## 17. 최종 결론

이번 프로젝트의 핵심은 다음 세 가지로 정리한다.

### 1. 에너지 소비량 예측
주행·환경 조건을 기반으로 `energy_consumption_kwhper100km`를 예측한다.

### 2. 운행 조건별 시뮬레이션
속도, HVAC 사용 전력, 적재량 등 현실적으로 비교 가능한 조건을 변경하여 여러 시나리오의 예상 소비량을 비교한다.

### 3. 주요 관련 요인 분석
EDA와 모델 해석 결과를 이용하여 높은 에너지 소비량 예측과 관련성이 큰 변수를 설명한다.

---

### 최종 프로젝트 정의

> **전기차 주행 조건 데이터를 분석하여 에너지 소비량 예측 모델을 구축하고, 기업이 다양한 운행 조건에 따른 예상 에너지 소비량을 비교하여 효율적인 운행 계획을 검토할 수 있도록 하는 실습 프로젝트**

현재 데이터의 한계를 넘어 과도하게 기능을 확장하기보다, **EDA → 회귀 모델 학습·평가 → 모델 해석 → 운행 조건 시뮬레이션**까지의 흐름을 완성하는 것을 우선 목표로 한다.
